In [155]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import scipy as sp
import sklearn as sk
from datetime import datetime
from sklearn.neighbors import NearestNeighbors

In [ ]:
# date is input as a string in yyyy-mm-dd format! Data is just the DataFrame from the CSV
def historical_constituients(date, data):
    """
    Inputs:
    - date: the "as of" date
    - data: a DataFrame where each row has a date and a list of the S&P500 constituents at that time.
    Outputs:
    - tickers: the list of tickers that were in the S&P on the given date
    """
    target_date = pd.to_datetime(date).normalize()
    if target_date > data['date'].iloc[-1]:
        raise ValueError(f'{target_date} is after the most recent update!')

    if target_date < data['date'].iloc[0]:
        raise ValueError(f'{target_date} is before the earliest entry!')

    # most recent date that is before or on the target date (fixes the issue of target date being between 2 recorded entries)
    mask = data['date'] <= target_date
    row = data.loc[mask].iloc[-1]

    return(row['tickers'])

In [ ]:
"""
This function is dependent upon stock price data. This needs to be resolved before troubleshooting anything else.
"""

In [ ]:
"""
So this needs to be reworked -- I may ask someone more reliable with code to try their hand at this before I bungle it up - Andy

In hindsight, I was doing too much by trying to abstract all the way out to dates, especially since I wasn't wholly clear about how I was computing returns / market constituency. 
A better prototype of this function will do the following:

Take the tickers (constituency) as of a specific day and pull the returns data held on the "as of" day. I presume Jun Yen can alter the return type ("daily," "monthly," or "annual") in his queries. If this is not possible, I can revisit this problem.

This would change the I/O field to be:

    Inputs:
    - tickers: the tickers for the S&P consitutients "as of" the end date. These come from "historical consitutients"
    - date
    - data
    Outputs:
    - cov_matrix: the covariance matrix for demeaned daily returns during the price window.
    - demeaned_returns: the demeaned, daily returns of the stock tickers between the start and end dates 

"""




def covariance_matrix(tickers, start_date, end_date):
    """
    Inputs:
    - tickers: the tickers for the S&P consitutients "as of" the end date. These come from "historica consitutients"
    - start_date: the beginning date of the price window
    - end_date: the end date of the price window. Generally, the end date is going to be the same date that the list of tickers was pulled.
    Outputs:
    - cov_matrix: the covariance matrix for demeaned daily returns during the price window.
    - demeaned_returns: the demeaned, daily returns of the stock tickers between the start and end dates 
    """

    stock_data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=True)
    prices = stock_data.xs('Close', axis=1,level=1)
    returns = prices.pct_change().dropna()
    demeaned_returns = returns - returns.mean()
    n, p = demeaned_returns.shape
    cov_matrix = demeaned_returns.cov()
    return cov_matrix, demeaned_returns

In [ ]:
"""
The following functions are dependent upon a functioning covariance matrix
"""

In [157]:
def RMT(eigenvalues, demeaned_returns):
    """
    Inputs:
    - eigenvalues: the output from scipy eigh
    - - demeaned_returns: the "demeaned_returns" output from the covariance_matrix() function. It is the original data matrix
    Outputs:
    - dimensions: dimensionality estimate
    """
    X = demeaned_returns
    n, p = X.shape
    evals = np.sort(eigenvalues)[::-1] # from ascending to descending order
    q = p/n

    # estimates sigma_squared by taking the median of the bottom 50% of eigenvalues since they are dominated by the noise term
    noise_idx = p // 2 
    noise = evals[-noise_idx:]
    sigma_squared = np.median(noise)

    # Marchenko-Pastur upper noise bound
    upper_bound = sigma_squared * (1+ np.sqrt(q))**2
    dimensions = np.sum(evals > upper_bound)

    return dimensions
    

def participation_ratio(eigenvalues):
    """
    Inputs:
    - eigenvalues: the output from scipy eigh
    Outputs:
    - dimensions: dimensionality estimate
    """
    eval_sum = np.sum(eigenvalues)
    sq_sum = np.sum(eigenvalues**2)
    
    dimensions = (eval_sum**2)/sq_sum
    
    return dimensions

def levina_bickel(demeaned_returns, k):
    """
    Inputs:
    - demeaned_returns: the "demeaned_returns" output from the covariance_matrix() function. It is the original data matrix
    - k: number of nearest neighbors used
    
    Outputs:
    - dimensions: dimensionality estimate
    """
    X = demeaned_returns
    n, p = X.shape
    
    # get the k nearnest neighbors using Euclidean Distance
    neighbors = NearestNeighbors(n_neighbors=k, metric='euclidean')
    neighbors.fit(X)
    distances, indicies = neighbors.kneighbors(X)

    mle_estimates = np.zeros(n)
    
    for i in range(n):
        # The implemention of the result from Levina & Bickel (2004)
        T_k = distances[i:-1]
        T_j = distances[i,:]

        T_j = T_j[T_j > 0]

        log_ratio = np.log(T_k/T_j)

        mle_i = (len(T_j)-1) / np.sum(log_ratio)
        mle_estimates[i] = mle_i
    
    dimensions = np.mean(mle_estimates)
    return dimensions
        

In [ ]:
# Now I am going to build out the dataframe. The specifics of this function WILL need to be reworked as we implement Jun Yen's data, but I wanted to at least leave a blueprint
def dimensions_through_time(date_list, k, data):
    """
    Inputs:
    - demeaned_returns: the "demeaned_returns" output from the covariance_matrix() function. It is the original data matrix
    - k: number of nearest neighbors used
    
    Outputs:
    - dimensions: dimensionality estimate
    """
    
    entries = []
    for date in date_list:
        tickers = historical_constituients(date, data)
        # major disclaimer, I am assuming that the inputs to this function are altered by the time that we get to testing this bad boi
        cov_matrix, demeaned_returns = covariance_matrix(tickers, date, data) 

        eigenvalues = np.linalg.eigvals(cov_matrix)

        rmt_output = RMT(eigenvalues, demeaned_returns)
        pr_output = participation_ratio(eigenvalues)
        lb_output = levina_bickel(demeaned_returns, k)

        entry = {
            'Date': date, 
            'RMT':rmt_output, 
            'Participation Ratio':pr_output, 
            'Levina-Bickel': lb_output}

        entries.append(entry)
        
    df = pd.DataFrame(entries)
    df.set_index('date', inplace=True)

    return df
    

In [59]:
# Load the data file and let's get schmoovin'
data = pd.read_csv("S&P 500 Historical Components & Changes(01-17-2026).csv", parse_dates=['date'])